# Введение в MapReduce модель на Python


In [599]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [600]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [601]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [602]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [603]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [604]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [605]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [606]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [607]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [608]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [609]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [610]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [611]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [612]:
from typing import NamedTuple # requires python 3.6+

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [613]:
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.032398031111118)),
 (1, np.float64(2.032398031111118)),
 (2, np.float64(2.032398031111118)),
 (3, np.float64(2.032398031111118)),
 (4, np.float64(2.032398031111118))]

## Inverted index

In [614]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [615]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [616]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [617]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('is', 18), ('it', 18)]),
 (1, [('banana', 2), ('what', 10)])]

## TeraSort

In [618]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.005522117123602399)),
   (None, np.float64(0.02541912674409519)),
   (None, np.float64(0.03142918568673425)),
   (None, np.float64(0.06355835028602363)),
   (None, np.float64(0.07404465173409036)),
   (None, np.float64(0.10789142699330445)),
   (None, np.float64(0.11586905952512971)),
   (None, np.float64(0.1195942459383017)),
   (None, np.float64(0.3109823217156622)),
   (None, np.float64(0.32518332202674705)),
   (None, np.float64(0.3308980248526492)),
   (None, np.float64(0.3584657285442726)),
   (None, np.float64(0.42754101835854963)),
   (None, np.float64(0.4722149251619493)),
   (None, np.float64(0.49379559636439074))]),
 (1,
  [(None, np.float64(0.5227328293819941)),
   (None, np.float64(0.5612771975694962)),
   (None, np.float64(0.6232981268275579)),
   (None, np.float64(0.6364104112637804)),
   (None, np.float64(0.6375574713552131)),
   (None, np.float64(0.7068573438476171)),
   (None, np.float64(0.713244787222995)),
   (None, np.float64(0.729007168

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [619]:
numbers = [5, 2, 9, 1, 7, 3]

def RECORDREADER():
    for i, value in enumerate(numbers):
        yield (i, value)

def MAP(_, value):
    yield ("max", value)

def REDUCE(_, values:Iterator[int]):
  max_val = float('-inf')
  for v in values:
    if v > max_val:
      max_val = v
  yield ('max', max_val)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))
print("Проверка:", max(numbers))

[('max', 9)]
Проверка: 9


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [620]:
numbers = [5, 2, 9, 1, 7, 3]

def RECORDREADER():
    for i, value in enumerate(numbers):
        yield (i, value)

def MAP(_, value):
    yield ("avg", (value, 1))

def REDUCE(key, values):
    total = 0
    count = 0
    for s, c in values:
        total += s
        count += c
    yield (key, total / count)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))
print("Проверка:", sum(numbers)/len(numbers))

[('avg', 4.5)]
Проверка: 4.5


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [621]:
class User(NamedTuple):
    id: int
    age: str
    social_contacts: int
    gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAP(_, row):
    if (row.gender == 'female'):
        yield (row.age, row)

def groupbykey_sorted(iterable):
    sorted_pairs = sorted(iterable, key=lambda x: x[0])

    res = []
    current_key = None
    current_values = []

    for key, val in sorted_pairs:
        if key == current_key:
            current_values.append(val)
        else:
            if current_key is not None:
                res.append((current_key, current_values))
            current_key = key
            current_values = [val]

    if current_key is not None:
        res.append((current_key, current_values))

    return res

shuffle_output = groupbykey_sorted(list(flatten(map(lambda x: MAP(*x), RECORDREADER()))))
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [622]:
import random

def generate_data_with_duplicates(n=100):
    elements = ['A', 'B', 'C', 'D', 'E',
              'F', 'G', 'H', 'I', 'J']
    data = []
    for _ in range(n):
        if random.random() < 0.3 and data:
            data.append(random.choice(data))
        else:
            data.append(random.choice(elements))
    return data

data = generate_data_with_duplicates(100)
maps = 3
reducers = 2

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for idx, val in enumerate(split):
            yield (idx, val)

    split_size =  int(np.ceil(len(data)/maps))
    for i in range(0, len(data), split_size):
        yield RECORDREADER(data[i:i+split_size])

def MAP(_, val: str):
    yield(val, None)

def REDUCE(key, values):
    yield (key, None)

def PARTITIONER(key):
    global reducers
    return hash(key) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None)
result = []
for _, partition in partitioned_output:
    result.extend([key for (key, _) in partition])
print("Уникальные элементы", sorted(set(result)))
print("Проверка: ", sorted(set(data)))

100 key-value pairs were sent over a network.
Уникальные элементы ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
Проверка:  ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [623]:
def generate_users(n=50):
    users = []
    genders = ['male', 'female']
    for i in range(n):
        users.append(User(
            id=i,
            age=random.randint(18, 60),
            social_contacts=random.randint(0, 1000),
            gender=random.choice(genders)
        ))
    return users

data = generate_users(100)
maps = 3
reducers = 2

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for user in split:
            yield (user.id, user)

    split_size = int(np.ceil(len(data)/maps))
    for i in range(0, len(data), split_size):
        yield RECORDREADER(data[i:i+split_size])

def MAP(key, user: User):
    if user.gender == 'female':
        yield (user, user)

def REDUCE(key, values):
    yield (key, None)

def PARTITIONER(key):
    global reducers
    return hash(key.id) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None)

result = []
for _, partition in partitioned_output:
    for (user, _) in partition:
        result.append(user)

print(f"Всего пользователей: {len(data)}")
print(f"Женщин: {len(result)}")
print(f"Проверка: {len([u for u in data if u.gender == 'female'])}")

print("\nПервые 5 выбранных пользователей:")
for user in result[:5]:
    print(f"    id: {user.id}, возраст: {user.age}, пол: {user.gender}, контакты: {user.social_contacts}")

48 key-value pairs were sent over a network.
Всего пользователей: 100
Женщин: 48
Проверка: 48

Первые 5 выбранных пользователей:
    id: 2, возраст: 54, пол: female, контакты: 77
    id: 8, возраст: 37, пол: female, контакты: 697
    id: 14, возраст: 48, пол: female, контакты: 241
    id: 16, возраст: 27, пол: female, контакты: 958
    id: 26, возраст: 45, пол: female, контакты: 767


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [624]:
def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for user in split:
            yield (user.id, user)

    split_size = int(np.ceil(len(data)/maps))
    for i in range(0, len(data), split_size):
        yield RECORDREADER(data[i:i+split_size])


def MAP(key, user: User):
    projected = (user.gender, user.age)
    yield (projected, projected)

def REDUCE(key, values):
    yield (key, None)

def PARTITIONER(key):
    global reducers
    return hash(key[0]) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None)

result = []
for _, partition in partitioned_output:
    for (user, _) in partition:
        result.append(user)

print(f"Всего пользователей: {len(data)}")
print(f"Результатов проекции: {len(result)}")
expected_unique = {(user.gender, user.age) for user in data}
print("Проверка: ", len(expected_unique))

print("\nПервые 10 проекций:")
for proj in result[:10]:
    print(f"    пол: {proj[0]}, возраст: {proj[1]}")

100 key-value pairs were sent over a network.
Всего пользователей: 100
Результатов проекции: 60
Проверка:  60

Первые 10 проекций:
    пол: female, возраст: 18
    пол: female, возраст: 20
    пол: female, возраст: 21
    пол: female, возраст: 22
    пол: female, возраст: 23
    пол: female, возраст: 24
    пол: female, возраст: 25
    пол: female, возраст: 26
    пол: female, возраст: 27
    пол: female, возраст: 29


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [625]:
class User(NamedTuple):
    id: int
    name: str
    department: str

def generate_employees(dept, n=30):
    employees = []
    names = ['Brie', 'Susan', 'Linete', 'Gabriel', 'Tom', 'Mike', 'Carlos', 'Pol', 'Felicia', 'Idy']
    for i in range(n):
        employees.append(User(
            id=i + (100 if dept == 'IT' else 200),
            name=random.choice(names),
            department=dept
        ))
    return employees

it_dept = generate_employees('IT', 30)
hr_dept = generate_employees('HR', 30)

common_employees = [
    User(id=300, name="Felicia", department="Both"),
    User(id=301, name="Gabriel", department="Both"),
    User(id=302, name="Linete", department="Both"),
    User(id=303, name="Mike", department="Both"),
    User(id=304, name="Idy", department="Both")]


it_dept.extend(common_employees)
hr_dept.extend(common_employees)

data_collections = [it_dept, hr_dept]
maps = 4
reducers = 3

def INPUTFORMAT():
    global maps

    def RECORDREADER(collection):
        for employee in collection:
            yield (employee.id, employee)

    all_data = []
    for collection in data_collections:
        all_data.extend(collection)

    split_size = int(np.ceil(len(all_data)/maps))
    for i in range(0, len(all_data), split_size):
        yield RECORDREADER(all_data[i:i+split_size])

def MAP(key, employee: User):
    yield (employee, employee)

def REDUCE(key, values):
    yield (key, None)

def PARTITIONER(key):
    global reducers
    return hash(key.id) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE,PARTITIONER, COMBINER=None)

union_result = []
for _, partition in partitioned_output:
    for (employee, _) in partition:
        union_result.append(employee)

print(f"Число сотрудников: {len(union_result)}")
print(f"Ожидаемое количество: {len(set(it_dept + hr_dept))}")
print(f"Удалено дубликатов: {len(it_dept) + len(hr_dept) - len(union_result)}")
print(f"Id найденных сотрудников: {sorted([e.id for e in union_result])}")

70 key-value pairs were sent over a network.
Число сотрудников: 65
Ожидаемое количество: 65
Удалено дубликатов: 5
Id найденных сотрудников: [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 300, 301, 302, 303, 304]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [626]:
def INPUTFORMAT():
    global maps

    def RECORDREADER(split_data):
        for source_id, employee in split_data:
            yield (employee.id, (source_id, employee))


    mapper_inputs = []
    for source_id, collection in enumerate([it_dept, hr_dept]):
        for employee in collection:
            mapper_inputs.append((source_id, employee))

    split_size = int(np.ceil(len(mapper_inputs)/maps))
    for i in range(0, len(mapper_inputs), split_size):
        yield RECORDREADER(mapper_inputs[i:i+split_size])

def MAP(key, value):
    source_id, employee = value
    yield (employee, source_id)

def REDUCE(key, values):
    sources = set(values)

    if len(sources) == 2:
        yield (key, None)

def PARTITIONER(key):
    global reducers
    return hash(key.id) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None)

intersection_result = []
for _, partition in partitioned_output:
    for (employee, _) in partition:
        intersection_result.append(employee)

print(f"Найдено в пересечении: {len(intersection_result)}")
print(f"Ожидаемое пересечение: 5")
print(f"Id найденных сотрудников: {sorted([e.id for e in intersection_result])}")

70 key-value pairs were sent over a network.
Найдено в пересечении: 5
Ожидаемое пересечение: 5
Id найденных сотрудников: [300, 301, 302, 303, 304]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [627]:
def INPUTFORMAT():
    global maps

    def RECORDREADER(split_data):
        for source_id, employee in split_data:
            yield (employee.id, (source_id, employee))

    mapper_inputs = []
    for source_id, collection in enumerate([it_dept, hr_dept]):
        for employee in collection:
            mapper_inputs.append((source_id, employee))

    split_size = int(np.ceil(len(mapper_inputs) / maps))
    for i in range(0, len(mapper_inputs), split_size):
        yield RECORDREADER(mapper_inputs[i:i + split_size])

def MAP(key, value):
    source_id, employee = value
    yield (employee, source_id)

def REDUCE(key, values):
    sources = set(values)
    if sources == {0}:
        yield (key, None)

def PARTITIONER(key):
    global reducers
    return hash(key.id) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE,PARTITIONER, COMBINER=None)

difference_result = []
for _, partition in partitioned_output:
    for (employee, _) in partition:
        difference_result.append(employee)

print(f"Найдено: {len(difference_result)}")
print(f"Ожидаемое значение: 30")
print(f"Id найденных сотрудников: {sorted([e.id for e in difference_result])}")

70 key-value pairs were sent over a network.
Найдено: 30
Ожидаемое значение: 30
Id найденных сотрудников: [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [628]:
class Student(NamedTuple):
    student_id: int
    group_id: int
    name: str

class Group(NamedTuple):
    group_id: int
    group_name: str
    faculty: str

def generate_students(n=50):
    students = []
    names = ['Brie', 'Susan', 'Linete', 'Gabriel', 'Tom', 'Mike', 'Carlos', 'Pol', 'Felicia', 'Idy']
    for i in range(n):
        students.append(Student(
            student_id=1000 + i,
            group_id=random.randint(1, 10),
            name=random.choice(names)
        ))
    return students

def generate_groups():
    groups = []
    faculties = ['Chemistry', 'Math', 'Physics', 'Biology']
    for i in range(1, 11):
        groups.append(Group(
            group_id=i,
            group_name=f"Group_{i}",
            faculty=random.choice(faculties)
        ))
    return groups

extra_groups = [
    Group(group_id=11, group_name="Group_11", faculty="Chemistry"),
    Group(group_id=12, group_name="Group_12", faculty="Math")
]

students = generate_students(50)
groups = generate_groups() + extra_groups

maps = 3
reducers = 2

def INPUTFORMAT():
    global maps

    def RECORDREADER(split_data):
        for rel_name, record in split_data:
            if rel_name == 'R':
                yield (record.group_id, ('R', record))
            else:
                yield (record.group_id, ('S', record))

    mapper_inputs = []
    for student in students:
        mapper_inputs.append(('R', student))
    for group in groups:
        mapper_inputs.append(('S', group))

    split_size = int(np.ceil(len(mapper_inputs) / maps))
    for i in range(0, len(mapper_inputs), split_size):
        yield RECORDREADER(mapper_inputs[i:i + split_size])

def MAP(key, value):
    rel_name, record = value
    yield (key, (rel_name, record))

def REDUCE(key, values):
    r_values = []
    s_values = []

    for rel_name, record in values:
        if rel_name == 'R':
            r_values.append(record)
        else:
            s_values.append(record)

    for student in r_values:
        for group in s_values:
            result = {
                'student_id': student.student_id,
                'student_name': student.name,
                'group_id': key,
                'group_name': group.group_name,
                'faculty': group.faculty
            }
            yield (key, result)

def PARTITIONER(key):
    global reducers
    return hash(key) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None)

join_result = []
for _, partition in partitioned_output:
    for (_, result) in partition:
        join_result.append(result)

print(f"Записей: {len(join_result)}")
print(f"Ожидаемое значение: {len(students)}")

print("\nРезультат (первые 5):")
for res in join_result[:5]:
    print(f"    {res['student_name']} (id: {res['student_id']}) -> "
          f"{res['group_name']} ({res['faculty']})")

62 key-value pairs were sent over a network.
Записей: 50
Ожидаемое значение: 50

Результат (первые 5):
    Tom (id: 1003) -> Group_2 (Chemistry)
    Linete (id: 1014) -> Group_2 (Chemistry)
    Tom (id: 1024) -> Group_2 (Chemistry)
    Carlos (id: 1027) -> Group_2 (Chemistry)
    Idy (id: 1033) -> Group_2 (Chemistry)


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [629]:
class Sale(NamedTuple):
    department: str
    price: float
    product: str

def generate_sales(n: int):
    departments = ['Electronics', 'Clothing', 'Books', 'Home', 'Sports', 'Toys']
    products = {
        'Electronics': ['Laptop', 'Phone', 'Tablet', 'Headphones'],
        'Clothing': ['Shirt', 'Jeans', 'Dress'],
        'Books': ['Novel', 'Detective', 'Fantasy'],
        'Home': ['Lamp', 'Chair', 'Table'],
        'Sports': ['Ball', 'Swimsuit', 'Shoes'],
        'Toys': ['Doll', 'Car', 'Plane']
    }

    sales = []

    for i in range(n):
        dept = random.choice(departments)
        price = round(random.uniform(10, 1000), 2)
        product = random.choice(products.get(dept, ['Generic']))

        sales.append(Sale(
            department=dept,
            price=price,
            product=product
        ))

    return sales

sales_data = generate_sales(200)

maps = 3
reducers = 2

def INPUTFORMAT():
    global maps

    def RECORDREADER(split_data):
        for sale in split_data:
            yield (sale.department, sale.price)

    split_size = int(np.ceil(len(sales_data) / maps))
    for i in range(0, len(sales_data), split_size):
        yield RECORDREADER(sales_data[i:i + split_size])

def MAP(key, price):
    yield (key, price)

def REDUCE_SUM(key, values):
    total = sum(values)
    yield (key, {'operation': 'SUM', 'result': total})

def REDUCE_COUNT(key, values):
    count = len(list(values))
    yield (key, {'operation': 'COUNT', 'result': count})

def REDUCE_AVG(key, values):
    values_list = list(values)
    avg = sum(values_list) / len(values_list) if values_list else 0
    yield (key, {'operation': 'AVG', 'result': round(avg, 2)})

def REDUCE_MAX(key, values):
    max_val = max(values)
    yield (key, {'operation': 'MAX', 'result': max_val})

def REDUCE_MIN(key, values):
    min_val = min(values)
    yield (key, {'operation': 'MIN', 'result': min_val})

def PARTITIONER(key):
    global reducers
    return hash(key) % reducers


partitioned_output_sum = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE_SUM, PARTITIONER, COMBINER=None)

sum_results = []
for _, partition in partitioned_output_sum:
    for (dept, result) in partition:
        sum_results.append((dept, result))

print("Сумма продаж по отделам:")
for dept, res in sorted(sum_results):
    print(f"  {dept}: {res['result']:.2f}")

partitioned_output_avg = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE_AVG, PARTITIONER, COMBINER=None)

avg_results = []
for _, partition in partitioned_output_avg:
    for (dept, result) in partition:
        avg_results.append((dept, result))

print("Средняя сумма продажи по отделам:")
for dept, res in sorted(avg_results):
    print(f"  {dept}: {res['result']:.2f}")

200 key-value pairs were sent over a network.
Сумма продаж по отделам:
  Books: 12952.89
  Clothing: 15957.50
  Electronics: 20942.37
  Home: 17523.56
  Sports: 20420.94
  Toys: 14426.80
200 key-value pairs were sent over a network.
Средняя сумма продажи по отделам:
  Books: 479.74
  Clothing: 483.56
  Electronics: 510.79
  Home: 531.02
  Sports: 583.46
  Toys: 465.38


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [630]:
from collections import defaultdict

shape = (5, 4)
m = np.ones(shape)
v = np.random.rand(4)

expected_result = np.dot(m, v)

def RECORDREADER():
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            yield (('M', i, j), m[i, j])
    for j in range(v.shape[0]):
        yield (('V', j), v[j])

def MAP(key, value):
    if key[0] == 'M':
        yield (key[2], ('M', key[1], value))
    else:
        yield (key[1], ('V', value))

def REDUCE(key, values):
    vector_val = 0
    matrix_cells = []

    for v in values:
        if v[0] == 'V':
            vector_val = v[1]
        else:
            matrix_cells.append((v[1], v[2]))

    for (i, m_val) in matrix_cells:
        yield (i, m_val * vector_val)

mr_output = list(MapReduce(RECORDREADER, MAP, REDUCE))

result = defaultdict(float)
for i, val in mr_output:
    result[i] += val

result_vector = [result[i] for i in range(shape[0])]

print("Результат: ", result_vector)
print("Ожидалось: ", expected_result)
print("Проверка: ", np.allclose(expected_result, result_vector))

Результат:  [np.float64(1.9797853753159975), np.float64(1.9797853753159975), np.float64(1.9797853753159975), np.float64(1.9797853753159975), np.float64(1.9797853753159975)]
Ожидалось:  [1.97978538 1.97978538 1.97978538 1.97978538 1.97978538]
Проверка:  True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [631]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [632]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1  # это n_jk

    # для каждого i берём m_ij
    for i in range(I):
        mij = small_mat[i, j]
        if mij != 0:
            yield ((i, k), mij * w)

def REDUCE(key, values):
    (i, k) = key

    total = 0
    for value in values:
        total += value

    yield ((i, k), total)

Проверьте своё решение

In [633]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [634]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [635]:
I, J, K = 2, 3, 4
A = np.random.rand(I, J)
B = np.random.rand(J, K)

def RECORDREADER():
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            yield (('A', i, j), A[i, j])
    for j in range(B.shape[0]):
        for k in range(B.shape[1]):
            yield (('B', j, k), B[j, k])

def MAP(key, value):
    if key[0] == 'A':
        i, j = key[1], key[2]
        for k in range(K):
            yield ((i, k), ('A', j, value))

    elif key[0] == 'B':
        j, k = key[1], key[2]
        for i in range(I):
            yield ((i, k), ('B', j, value))

def REDUCE(key, values):
    row_A = {}
    col_B = {}

    for item in values:
        tag, j, val = item
        if tag == 'A':
            row_A[j] = val
        else:
            col_B[j] = val

    result = 0.0
    for j in range(J):
        a_val = row_A.get(j, 0.0)
        b_val = col_B.get(j, 0.0)
        result += a_val * b_val

    yield (key, result)

expected_result = np.matmul(A, B)
res = list(MapReduce(RECORDREADER, MAP, REDUCE))

result = np.zeros((I, K))
for (idx, val) in res:
    result[idx] = val

print(np.allclose(expected_result, result))

True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [636]:
I, J, K = 2, 3, 4
maps = 3
reducers = 2

A = np.random.rand(I, J)
B = np.random.rand(J, K)

input_values = []
for i in range(I):
    for j in range(J):
        input_values.append( (('A', i, j), A[i, j]) )
for j in range(J):
    for k in range(K):
        input_values.append( (('B', j, k), B[j, k]) )


def INPUTFORMAT():
    global maps
    split_size = int(np.ceil(len(input_values)/maps))
    for i in range(0, len(input_values), split_size):
        yield input_values[i : i + split_size]

def RECORDREADER(split):
    for item in split:
        yield item

def MAP(key, value):
    matrix_label = key[0]

    if matrix_label == 'A':
        i, j = key[1], key[2]
        for k in range(K):
            yield ((i, k), ('A', j, value))

    elif matrix_label == 'B':
        j, k = key[1], key[2]
        for i in range(I):
            yield ((i, k), ('B', j, value))

def REDUCE(key, values):
    row_A = {}
    col_B = {}

    for tag, j, val in values:
        if tag == 'A':
            row_A[j] = val
        else:
            col_B[j] = val

    result_val = 0.0
    for j in range(J):
        a = row_A.get(j, 0.0)
        b = col_B.get(j, 0.0)
        result_val += a * b

    yield (key, result_val)

def PARTITIONER(key):
    global reducers
    return hash(key) % reducers

res = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
result = np.zeros((I, K))

for partition_id, partition_list in res:
    for key, value in partition_list:
        i, k = key
        result[i, k] = value

print(np.allclose(np.matmul(A, B), result))

48 key-value pairs were sent over a network.
True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [637]:
subset_size = int(len(input_values) * 0.7)
random_subset_input = random.sample(input_values, subset_size)

def INPUTFORMAT():
    global maps
    split_size = int(np.ceil(len(random_subset_input)/maps))
    for i in range(0, len(random_subset_input), split_size):
        yield random_subset_input[i : i + split_size]

res = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
result = np.zeros((I, K))
for partition_id, partition_list in res:
    for key, value in partition_list:
        i, k = key
        result[i, k] = value

A_sparse = np.zeros((I, J))
B_sparse = np.zeros((J, K))

for item in random_subset_input:
    key, val = item
    tag = key[0]
    if tag == 'A':
        A_sparse[key[1], key[2]] = val
    else:
        B_sparse[key[1], key[2]] = val

expected_result = np.matmul(A_sparse, B_sparse)
print(np.allclose(expected_result, result))

34 key-value pairs were sent over a network.
True
